# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaant7/flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import os
from dotenv import load_dotenv
import duckdb
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

model_df = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position,
        SUM(ga4_sessions) AS total_sessions,
        SUM(scroll_events) AS total_scroll_events,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

model_df["is_declining"] = (model_df["imp_second_half"] < model_df["imp_first_half"]).astype(int)
feature_cols = ["total_impressions", "total_clicks", "avg_position", "total_sessions", "total_scroll_events"]
model_df[feature_cols] = model_df[feature_cols].fillna(0)
model_df["ctr"] = 100.0 * model_df["total_clicks"] / model_df["total_impressions"]

In [2]:
np.random.seed(42)
all_clients = model_df["client_hash_id"].unique()
n_holdout = int(len(all_clients) * 0.20)
holdout_clients = set(np.random.choice(all_clients, size=n_holdout, replace=False))

train_df = model_df[~model_df["client_hash_id"].isin(holdout_clients)]
test_df = model_df[model_df["client_hash_id"].isin(holdout_clients)].copy()

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(train_df[feature_cols], train_df["is_declining"])
test_df["model_prob"] = rf.predict_proba(test_df[feature_cols])[:, 1]

print("Honest AUC on holdout:", roc_auc_score(test_df["is_declining"], test_df["model_prob"]))

Honest AUC on holdout: 0.5048933609851689


In [3]:
# Baseline flag from Week 4 (low_ctr_visible_page)
test_df["baseline_flag"] = (
    (test_df["total_impressions"] >= 500) &
    (test_df["avg_position"] > 0) & (test_df["avg_position"] <= 20) &
    (test_df["ctr"] < 0.5)
).astype(int)

# Blend: mostly model, some baseline — following the reference pipeline's pattern
test_df["playbook_score"] = 100 * (0.70 * test_df["model_prob"] + 0.30 * test_df["baseline_flag"])

def assign_reason_code(row):
    if row["baseline_flag"] == 1 and row["model_prob"] >= 0.5:
        return "model_and_ctr_flag"
    elif row["model_prob"] >= 0.5:
        return "model_decline_risk"
    elif row["baseline_flag"] == 1:
        return "low_ctr_visible_page"
    else:
        return "low_priority"

def assign_action(reason):
    mapping = {
        "model_and_ctr_flag": "review_title_meta_and_refresh",
        "model_decline_risk": "review_for_refresh",
        "low_ctr_visible_page": "review_title_meta",
        "low_priority": "monitor",
    }
    return mapping[reason]

test_df["reason_code"] = test_df.apply(assign_reason_code, axis=1)
test_df["action"] = test_df["reason_code"].apply(assign_action)

queue = test_df.sort_values("playbook_score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

print("Queue shape:", queue.shape)
queue[["content_hash_id", "playbook_score", "reason_code", "action", "rank"]].head(15)

Queue shape: (18591, 17)


,content_hash_id,playbook_score,reason_code,action,rank
0,content_b215e9a73e2284ad,64.782209,low_ctr_visible_page,review_title_meta,1
1,content_cfcddd31c050406c,62.478317,low_ctr_visible_page,review_title_meta,2
2,content_7bbc06fb998c8b83,62.441211,low_ctr_visible_page,review_title_meta,3
3,content_96f4295b7b3218b3,62.348035,low_ctr_visible_page,review_title_meta,4
4,content_0bd6827080aaf5d6,62.333540,low_ctr_visible_page,review_title_meta,5
5,content_98a13ce5999c1223,62.310907,low_ctr_visible_page,review_title_meta,6
6,content_e53eb1ea08b8260b,62.291934,low_ctr_visible_page,review_title_meta,7
7,content_6b1b5450f74c5ce7,62.286776,low_ctr_visible_page,review_title_meta,8
8,content_257935b0475f10a1,62.208980,low_ctr_visible_page,review_title_meta,9
9,content_a8998878909cee77,62.184839,low_ctr_visible_page,review_title_meta,10


### Ranked actions + reason codes

**Score formula:** `playbook_score = 100 * (0.70 * model_probability + 0.30 * baseline_flag)`
— following the reference pipeline's blend pattern, weighting the model more heavily while
still keeping the transparent baseline rule as a component.

**Reason codes → actions:**

| Reason code | Meaning | Action |
|---|---|---|
| `model_and_ctr_flag` | Model flags decline risk AND meets the CTR-visibility rule | `review_title_meta_and_refresh` |
| `model_decline_risk` | Model probability ≥ 0.5 alone | `review_for_refresh` |
| `low_ctr_visible_page` | Meets the CTR-visibility rule, model doesn't flag it | `review_title_meta` |
| `low_priority` | Neither condition met | `monitor` |

**Honest AUC on this run's holdout: 0.505** — essentially indistinguishable from random
guessing, and weaker than the 0.547–0.576 range measured in Weeks 5–6. This is a direct,
visible consequence of the run-to-run variance flagged in Week 6's claim rewrite: model
performance is sensitive to which clients land in the holdout set, and this run landed on
the weak end. Because of this, the top of the ranked queue is dominated entirely by
`low_ctr_visible_page` — the model's contribution to ranking is negligible this run, and the
transparent baseline rule is effectively doing the ranking work.

**What this means for the playbook:** I'm treating the model as a **secondary, not primary**
signal in this version of the queue. The baseline rule remains the more trustworthy driver of
ranking given this run's weak model performance — a finding worth stating plainly rather than
dressing up the blend as more sophisticated than it currently is.

**Archetype → action mapping (descriptive, not a fixed taxonomy):**

- **High-visibility, low-CTR, well-positioned** (`low_ctr_visible_page`) → `review_title_meta` —
  the clearest, most measurable win: title/snippet review on pages that already earn impressions.
- **Model-flagged decline risk without a clear CTR gap** (`model_decline_risk`) → `review_for_refresh` —
  weaker evidence (AUC ~0.5), so this action should carry a lower confidence label than CTR-flagged pages.
- **Both signals agree** (`model_and_ctr_flag`) → `review_title_meta_and_refresh` — the strongest
  candidates, though rare this run given the weak model signal.
- **Neither signal fires** (`low_priority`) → `monitor` — no action needed now, revisit next cycle.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

**Who uses this, and how:** A content editor or SEO reviewer uses this ranked queue as a
starting point for their weekly/monthly review cycle — not as an automated action list. The
queue narrows a large inventory down to a manageable shortlist; a human still reads each
flagged page, confirms the reason code makes sense, and decides whether to act.

**What this playbook can say:**
- **Observed:** which pages show a measurable CTR gap for their position, and which pages
  the model associates (weakly) with decline, within the March 2026 window.
- **Directional:** pages meeting the `low_ctr_visible_page` rule are somewhat more likely to
  have title/snippet issues worth reviewing, based on the confirmed CTR-position relationship
  from Week 4.
- **Decision-support:** a starting shortlist for limited reviewer time, ranked by a
  transparent, inspectable score — not a final verdict on any single page.

**What this playbook cannot say:**
- **Not causal:** flagging a page does not mean fixing it will recover traffic — no experiment
  was run.
- **Not a strong prediction:** this run's model AUC was 0.505, essentially random. The
  `model_decline_risk` and `model_and_ctr_flag` actions should be treated with **low
  confidence** this cycle — the ranking is currently carried almost entirely by the
  transparent baseline rule, not the model.
- **Not validated beyond March 2026:** this queue reflects one month's snapshot on 47 client's
  worth of training data; it has not been tested on a different month or a larger client base.
- **Not a Google-algorithm claim:** nothing here reverse-engineers or proves how search
  ranking works — these are FlyRank's own observed signals.

**Where this breaks down:** The model's weak and unstable AUC (0.505–0.576 across three runs
in Weeks 5–7) means this playbook is currently closer to "a well-organized version of the
Week 4 baseline rule" than "a genuinely learned prioritization system." That's a fair, honest
description of where this capstone landed — not a failure to hide, but the actual, measured
state of the evidence.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.